# RNA-seq Quality Control and Alignment Summary
### *Anopheles coluzzii* — Ngousso colony vs field-collected mosquitoes (Akoda, Osun State, Nigeria)

**Imports and plotting configuration**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse, FancyBboxPatch
from pathlib import Path

plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.labelweight"] = "bold"
plt.rcParams["axes.titlesize"] = 17
plt.rcParams["axes.labelsize"] = 13
plt.rcParams["xtick.labelsize"] = 11.5
plt.rcParams["ytick.labelsize"] = 11.5
plt.rcParams["legend.fontsize"] = 11.5
plt.rcParams["figure.dpi"] = 150
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["axes.linewidth"] = 1.4
plt.rcParams["xtick.major.width"] = 1.4
plt.rcParams["ytick.major.width"] = 1.4

OUTDIR = Path("/mnt/hpc_acegid/home/khadmig/work/data/For_Lynda/260718_VH00635_9_AAJ2KFGM5/output_analysis")
MQC = OUTDIR / "multiqc" / "multiqc_data"
SALMON_DIR = OUTDIR / "salmon"
FIGDIR = Path("figures")
FIGDIR.mkdir(exist_ok=True)

SAMPLES = ["Ng1", "Ng2", "Ng3", "NE1", "NE2", "NE3"]
GROUP = {
    "Ng1": "Ngousso colony",
    "Ng2": "Ngousso colony",
    "Ng3": "Ngousso colony",
    "NE1": "Field (Akoda)",
    "NE2": "Field (Akoda)",
    "NE3": "Field (Akoda)",
}
GROUP_COLOR = {"Ngousso colony": "#1B4F72", "Field (Akoda)": "#B03A2E"}
SAMPLE_COLOR = {s: GROUP_COLOR[GROUP[s]] for s in SAMPLES}


def bold_ticks(ax):
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontweight("bold")


def legend_by_group(ax, loc="upper right"):
    handles = [plt.Rectangle((0, 0), 1, 1, color=GROUP_COLOR[g]) for g in GROUP_COLOR]
    ax.legend(handles, list(GROUP_COLOR.keys()), loc=loc, frameon=False)

**Pipeline workflow**

In [ ]:
stages = [
    {"name": "FastQC", "sub": "raw read QC", "pos": (0.7, 1.0), "color": "#3B6EA5"},
    {"name": "Trim Galore", "sub": "adapter / quality trimming", "pos": (3.15, 1.0), "color": "#3C9D8F"},
    {"name": "STAR", "sub": "genome + transcriptome\nalignment", "pos": (5.6, 1.0), "color": "#D4A017"},
    {"name": "samtools", "sub": "sort, index, flagstat", "pos": (8.3, 1.85), "color": "#8E5572"},
    {"name": "Salmon", "sub": "transcript quantification", "pos": (8.3, 0.15), "color": "#D9622B"},
    {"name": "MultiQC", "sub": "aggregated QC report", "pos": (11.0, 1.0), "color": "#4B4B4B"},
]

box_w, box_h = 2.15, 1.05

fig, ax = plt.subplots(figsize=(14.5, 10.8))

for stage in stages:
    x, y = stage["pos"]
    box = FancyBboxPatch(
        (x - box_w / 2, y - box_h / 2),
        box_w,
        box_h,
        boxstyle="round,pad=0.02,rounding_size=0.09",
        linewidth=1.6,
        edgecolor="white",
        facecolor=stage["color"],
    )
    ax.add_patch(box)
    ax.text(x, y + 0.3, stage["name"], ha="center", va="center", fontsize=14.5, fontweight="bold", color="white")
    ax.text(x, y - 0.14, stage["sub"], ha="center", va="center", fontsize=9.5, color="white", linespacing=1.6)


def arrow(p_from, p_to):
    ax.annotate(
        "",
        xy=p_to,
        xytext=p_from,
        arrowprops=dict(arrowstyle="-|>", lw=2.2, color="#2B2B2B", shrinkA=0, shrinkB=0),
    )


fastqc, trim, star, samt, salmon, mqc = stages
arrow((fastqc["pos"][0] + box_w / 2, fastqc["pos"][1]), (trim["pos"][0] - box_w / 2, trim["pos"][1]))
arrow((trim["pos"][0] + box_w / 2, trim["pos"][1]), (star["pos"][0] - box_w / 2, star["pos"][1]))
arrow((star["pos"][0] + box_w / 2, star["pos"][1] + 0.15), (samt["pos"][0] - box_w / 2, samt["pos"][1]))
arrow((star["pos"][0] + box_w / 2, star["pos"][1] - 0.15), (salmon["pos"][0] - box_w / 2, salmon["pos"][1]))
arrow((samt["pos"][0] + box_w / 2, samt["pos"][1]), (mqc["pos"][0] - box_w / 2, mqc["pos"][1] + 0.15))
arrow((salmon["pos"][0] + box_w / 2, salmon["pos"][1]), (mqc["pos"][0] - box_w / 2, mqc["pos"][1] - 0.15))

box_w2, box_h2 = 1.9, 0.95

downstream_stages = [
    {"name": "Gene aggregation", "sub": "transcript -> gene\n(GTF-based)", "pos": (8.3, -1.7), "color": "#5D6D7E"},
    {"name": "Biotype\nclassification", "sub": "coding / non-coding", "pos": (2.3, -3.2), "color": "#3B6EA5"},
    {"name": "PCA", "sub": "transcript + gene level", "pos": (5.05, -3.2), "color": "#3C9D8F"},
    {"name": "Differential\nexpression", "sub": "PyDESeq2, MA + volcano", "pos": (7.8, -3.2), "color": "#B03A2E"},
    {"name": "Overlap", "sub": "shared vs group-specific", "pos": (10.55, -3.2), "color": "#5B2C6F"},
    {"name": "Reports", "sub": "figures + Excel workbooks", "pos": (6.4, -4.6), "color": "#4B4B4B"},
]

for stage in downstream_stages:
    x, y = stage["pos"]
    box = FancyBboxPatch(
        (x - box_w2 / 2, y - box_h2 / 2),
        box_w2,
        box_h2,
        boxstyle="round,pad=0.02,rounding_size=0.09",
        linewidth=1.6,
        edgecolor="white",
        facecolor=stage["color"],
    )
    ax.add_patch(box)
    ax.text(x, y + 0.22, stage["name"], ha="center", va="center", fontsize=12, fontweight="bold", color="white", linespacing=1.3)
    ax.text(x, y - 0.22, stage["sub"], ha="center", va="center", fontsize=8.5, color="white", linespacing=1.5)

gene_agg, biotype_box, pca_box, de_box, overlap_box, reports_box = downstream_stages

arrow((salmon["pos"][0], salmon["pos"][1] - box_h / 2), (gene_agg["pos"][0], gene_agg["pos"][1] + box_h2 / 2))

for target in (biotype_box, pca_box, de_box, overlap_box):
    arrow((gene_agg["pos"][0], gene_agg["pos"][1] - box_h2 / 2), (target["pos"][0], target["pos"][1] + box_h2 / 2))

for source in (biotype_box, pca_box, de_box, overlap_box):
    arrow((source["pos"][0], source["pos"][1] - box_h2 / 2), (reports_box["pos"][0], reports_box["pos"][1] + box_h2 / 2))

ax.set_xlim(-0.8, 12.3)
ax.set_ylim(-5.3, 2.55)
ax.axis("off")
ax.set_title("RNA-seq processing and downstream analysis workflow", pad=14)

plt.tight_layout()
plt.savefig(FIGDIR / "01_workflow.png", bbox_inches="tight")
plt.show()

**Load QC summary tables**

In [ ]:
fastqc = pd.read_csv(MQC / "multiqc_fastqc.txt", sep="\t")
cutadapt = pd.read_csv(MQC / "multiqc_cutadapt.txt", sep="\t")
star = pd.read_csv(MQC / "multiqc_star.txt", sep="\t")
salmon = pd.read_csv(MQC / "multiqc_salmon.txt", sep="\t")

file_pattern = r"^(?P<sample>.+?)_S\d+_R(?P<mate>[12])_001$"

fastqc[["sample", "mate"]] = fastqc["Sample"].str.extract(file_pattern)
cutadapt[["sample", "mate"]] = cutadapt["Sample"].str.extract(file_pattern)
star = star[~star["Sample"].str.contains("STARpass1")].rename(columns={"Sample": "sample"}).set_index("sample").loc[SAMPLES]
salmon = salmon.rename(columns={"Sample": "sample"}).set_index("sample").loc[SAMPLES]

**Sequencing depth and read quality**

In [ ]:
depth = fastqc.groupby("sample").agg(
    read_pairs=("Total Sequences", "first"),
    percent_gc=("%GC", "mean"),
    percent_duplication=("total_deduplicated_percentage", lambda x: 100 - x.mean()),
    mean_length=("avg_sequence_length", "mean"),
).loc[SAMPLES]

trimming = cutadapt.groupby("sample").agg(percent_trimmed=("percent_trimmed", "mean")).loc[SAMPLES]

summary_table = depth.join(trimming)
summary_table.insert(0, "group", [GROUP[s] for s in summary_table.index])
summary_table = summary_table.rename(columns={
    "group": "Group",
    "read_pairs": "Read pairs",
    "percent_gc": "GC content (%)",
    "percent_duplication": "Duplication (%)",
    "mean_length": "Mean read length (bp)",
    "percent_trimmed": "Bases trimmed (%)",
})

styled_summary = (
    summary_table.style
    .format({
        "Read pairs": "{:,.0f}",
        "GC content (%)": "{:.1f}",
        "Duplication (%)": "{:.1f}",
        "Mean read length (bp)": "{:.1f}",
        "Bases trimmed (%)": "{:.2f}",
    })
    .background_gradient(subset=["Duplication (%)"], cmap="Reds")
    .background_gradient(subset=["Read pairs"], cmap="Blues")
    .set_caption("Sequencing depth and read quality per sample")
    .set_table_styles([
        {"selector": "caption", "props": [("font-size", "15px"), ("font-weight", "bold"), ("text-align", "left"), ("padding-bottom", "8px")]},
        {"selector": "th", "props": [("font-weight", "bold"), ("background-color", "#F2F2F2")]},
    ])
)
styled_summary

**Read pairs per sample**

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5))
bars = ax.bar(SAMPLES, summary_table["Read pairs"] / 1e6, color=[SAMPLE_COLOR[s] for s in SAMPLES], width=0.62)

for bar, value in zip(bars, summary_table["Read pairs"] / 1e6):
    ax.text(bar.get_x() + bar.get_width() / 2, value + 1, f"{value:.1f}", ha="center", fontsize=11, fontweight="bold")

ax.set_ylabel("Read pairs (millions)")
ax.set_title("Sequencing depth per sample")
ax.set_ylim(0, summary_table["Read pairs"].max() / 1e6 * 1.18)
ax.legend(
    handles=[plt.Rectangle((0, 0), 1, 1, color=GROUP_COLOR[g]) for g in GROUP_COLOR],
    labels=list(GROUP_COLOR.keys()),
    loc="upper center",
    bbox_to_anchor=(0.5, -0.12),
    ncol=2,
    frameon=False,
)
bold_ticks(ax)

plt.tight_layout()
plt.savefig(FIGDIR / "02_read_depth.png", bbox_inches="tight")
plt.show()

**Alignment rate to the *Anopheles coluzzii* genome**

In [ ]:
categories = [
    ("uniquely_mapped", "Uniquely mapped", "#1B4F72"),
    ("multimapped", "Multi-mapped", "#2E86AB"),
    ("multimapped_toomany", "Mapped to too many loci", "#E0A72E"),
    ("unmapped_tooshort", "Unmapped: too short", "#B03A2E"),
    ("unmapped_other", "Unmapped: other", "#9E9E9E"),
]

alignment_pct = pd.DataFrame({
    label: star[col] / star["total_reads"] * 100
    for col, label, _ in categories
}, index=SAMPLES)

fig, ax = plt.subplots(figsize=(8.5, 5.5))
bottom = np.zeros(len(SAMPLES))
for col, label, color in categories:
    values = alignment_pct[label].values
    ax.bar(SAMPLES, values, bottom=bottom, label=label, color=color, width=0.62)
    bottom += values

ax.set_ylabel("Reads (%)")
ax.set_ylim(0, 100)
ax.set_title("STAR alignment rate breakdown")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=3, frameon=False)
bold_ticks(ax)

plt.tight_layout()
plt.savefig(FIGDIR / "03_star_alignment.png", bbox_inches="tight")
plt.show()

**Fragments assigned to transcripts (Salmon)**

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5))
values = salmon.loc[SAMPLES, "num_mapped"] / 1e6
bars = ax.bar(SAMPLES, values, color=[SAMPLE_COLOR[s] for s in SAMPLES], width=0.62)

for bar, value in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, value + 0.4, f"{value:.1f}", ha="center", fontsize=11, fontweight="bold")

ax.set_ylabel("Assigned fragments (millions)")
ax.set_ylim(0, values.max() * 1.18)
ax.set_title("Salmon transcript quantification depth")
ax.legend(
    handles=[plt.Rectangle((0, 0), 1, 1, color=GROUP_COLOR[g]) for g in GROUP_COLOR],
    labels=list(GROUP_COLOR.keys()),
    loc="upper center",
    bbox_to_anchor=(0.5, -0.12),
    ncol=2,
    frameon=False,
)
bold_ticks(ax)

plt.tight_layout()
plt.savefig(FIGDIR / "04_salmon_mapping.png", bbox_inches="tight")
plt.show()

**Sample-to-sample similarity from transcript quantification**

In [ ]:
tpm = {}
for sample in SAMPLES:
    quant = pd.read_csv(SALMON_DIR / sample / "quant.sf", sep="\t")
    tpm[sample] = quant.set_index("Name")["TPM"]

tpm_matrix = pd.DataFrame(tpm)
log_tpm = np.log2(tpm_matrix + 1)
correlation = log_tpm.corr().loc[SAMPLES, SAMPLES]

triangle_mask = np.triu(np.ones(correlation.shape, dtype=bool), k=1)
correlation_masked = np.ma.masked_where(triangle_mask, correlation.values)

fig, ax = plt.subplots(figsize=(7, 6))
cmap = plt.get_cmap("viridis").copy()
cmap.set_bad("white")
im = ax.imshow(correlation_masked, cmap=cmap, vmin=0.90, vmax=1.0)

ax.set_xticks(range(len(SAMPLES)))
ax.set_yticks(range(len(SAMPLES)))
ax.set_xticklabels(SAMPLES, rotation=45, ha="right")
ax.set_yticklabels(SAMPLES)

for label, sample in zip(ax.get_xticklabels(), SAMPLES):
    label.set_color(SAMPLE_COLOR[sample])
for label, sample in zip(ax.get_yticklabels(), SAMPLES):
    label.set_color(SAMPLE_COLOR[sample])

for i in range(len(SAMPLES)):
    for j in range(len(SAMPLES)):
        if j > i:
            continue
        value = correlation.values[i, j]
        text_color = "white" if value < 0.965 else "black"
        ax.text(j, i, f"{value:.2f}", ha="center", va="center", color=text_color, fontsize=10.5, fontweight="bold")

bold_ticks(ax)
ax.set_title("Pairwise transcriptome correlation (log2 TPM)")
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Pearson correlation", fontweight="bold")

plt.tight_layout()
plt.savefig(FIGDIR / "05_sample_correlation.png", bbox_inches="tight")
plt.show()

**Shared and group-specific transcripts**

In [ ]:
from matplotlib.patches import Circle

TABLEDIR = Path("tables")
TABLEDIR.mkdir(exist_ok=True)

DETECTION_TPM = 1.0
MIN_REPLICATES = 2

ngousso_samples = [s for s in SAMPLES if GROUP[s] == "Ngousso colony"]
field_samples = [s for s in SAMPLES if GROUP[s] == "Field (Akoda)"]

detected = tpm_matrix >= DETECTION_TPM
ngousso_detected = detected[ngousso_samples].sum(axis=1) >= MIN_REPLICATES
field_detected = detected[field_samples].sum(axis=1) >= MIN_REPLICATES

category = np.select(
    [ngousso_detected & field_detected, ngousso_detected & ~field_detected, ~ngousso_detected & field_detected],
    ["Shared", "Ngousso colony only", "Field (Akoda) only"],
    default="Not detected",
)

transcript_table = tpm_matrix.round(2).copy()
transcript_table.insert(0, "category", category)
transcript_table = transcript_table.reset_index().rename(columns={"Name": "transcript_id"})

category_order = ["Shared", "Ngousso colony only", "Field (Akoda) only", "Not detected"]
category_counts = transcript_table["category"].value_counts().reindex(category_order).fillna(0).astype(int)
category_counts

**Export transcript categories to Excel**

In [ ]:
excel_path = TABLEDIR / "shared_vs_group_specific_transcripts.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    category_counts.rename("transcript_count").to_frame().to_excel(writer, sheet_name="summary")
    transcript_table.to_excel(writer, sheet_name="transcripts", index=False)

excel_path

**Overlap between Ngousso colony and field-collected transcript sets**

In [ ]:
n_shared = category_counts["Shared"]
n_ngousso_only = category_counts["Ngousso colony only"]
n_field_only = category_counts["Field (Akoda) only"]

fig, ax = plt.subplots(figsize=(8, 6.5))

left_circle = Circle((-0.72, 0), 1.35, color=GROUP_COLOR["Ngousso colony"], alpha=0.55, linewidth=2, edgecolor="white")
right_circle = Circle((0.72, 0), 1.35, color=GROUP_COLOR["Field (Akoda)"], alpha=0.55, linewidth=2, edgecolor="white")
ax.add_patch(left_circle)
ax.add_patch(right_circle)

ax.text(-1.35, 0, f"{n_ngousso_only:,}", ha="center", va="center", fontsize=20, fontweight="bold")
ax.text(1.35, 0, f"{n_field_only:,}", ha="center", va="center", fontsize=20, fontweight="bold")
ax.text(0, 0, f"{n_shared:,}", ha="center", va="center", fontsize=20, fontweight="bold")

ax.text(-0.72, 1.62, "Ngousso colony", ha="center", va="center", fontsize=13.5, fontweight="bold", color=GROUP_COLOR["Ngousso colony"])
ax.text(0.72, 1.62, "Field (Akoda)", ha="center", va="center", fontsize=13.5, fontweight="bold", color=GROUP_COLOR["Field (Akoda)"])

ax.set_xlim(-2.4, 2.4)
ax.set_ylim(-1.7, 2.1)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title("Transcript detection overlap (TPM ≥ 1 in ≥ 2/3 replicates)")

plt.tight_layout()
plt.savefig(FIGDIR / "06_transcript_overlap_venn.png", bbox_inches="tight")
plt.show()

**Transcript category counts**

In [ ]:
colors = [GROUP_COLOR["Ngousso colony"], GROUP_COLOR["Ngousso colony"], GROUP_COLOR["Field (Akoda)"], "#9E9E9E"]
colors[0] = "#5B2C6F"

fig, ax = plt.subplots(figsize=(8, 5.5))
bars = ax.bar(category_counts.index, category_counts.values, color=colors, width=0.6)

for bar, value in zip(bars, category_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, value + max(category_counts.values) * 0.015, f"{value:,}", ha="center", fontsize=11, fontweight="bold")

ax.set_ylabel("Number of transcripts")
ax.set_title("Transcript detection categories")
bold_ticks(ax)
plt.setp(ax.get_xticklabels(), rotation=12, ha="right")

plt.tight_layout()
plt.savefig(FIGDIR / "07_transcript_categories.png", bbox_inches="tight")
plt.show()

**Gene-level aggregation: parsing the GTF for transcript-to-gene mapping and biotypes**

In [ ]:
import re

def parse_gtf_biotypes(gtf_path):
    tx2gene = {}
    gene_biotype = {}
    transcript_biotype = {}
    with open(gtf_path) as handle:
        for line in handle:
            if line.startswith("#"):
                continue
            fields = line.rstrip("\n").split("\t")
            if len(fields) < 9 or fields[2] not in ("gene", "transcript"):
                continue
            attrs = fields[8]
            gene_match = re.search(r'gene_id "([^"]+)"', attrs)
            gid = gene_match.group(1) if gene_match else None
            if fields[2] == "gene":
                biotype_match = re.search(r'gene_biotype "([^"]+)"', attrs)
                gene_biotype[gid] = biotype_match.group(1) if biotype_match else "unknown"
            elif fields[2] == "transcript":
                transcript_match = re.search(r'transcript_id "([^"]+)"', attrs)
                tid = transcript_match.group(1) if transcript_match else None
                biotype_match = re.search(r'transcript_biotype "([^"]+)"', attrs)
                tx2gene[tid] = gid
                transcript_biotype[tid] = biotype_match.group(1) if biotype_match else "unknown"
    return tx2gene, gene_biotype, transcript_biotype

GTF_PATH = OUTDIR / "reference" / "Anopheles_coluzzii.AcolN3.63.gtf"
tx2gene, gene_biotype, transcript_biotype = parse_gtf_biotypes(GTF_PATH)

print(f"Genes annotated: {len(gene_biotype):,}")
print(f"Transcripts annotated: {len(transcript_biotype):,}")

**Building transcript-level and gene-level count and TPM matrices**

In [ ]:
raw_counts = {}
raw_tpm = {}
for sample in SAMPLES:
    quant = pd.read_csv(SALMON_DIR / sample / "quant.sf", sep="\t")
    raw_counts[sample] = quant.set_index("Name")["NumReads"]
    raw_tpm[sample] = quant.set_index("Name")["TPM"]

transcript_counts = pd.DataFrame(raw_counts).round(0).astype(int)
transcript_tpm = pd.DataFrame(raw_tpm)

gene_of_transcript = pd.Series(tx2gene).reindex(transcript_counts.index)
gene_counts = transcript_counts.groupby(gene_of_transcript).sum()
gene_tpm = transcript_tpm.groupby(gene_of_transcript).sum()

print(f"Transcript-level matrix: {transcript_counts.shape[0]:,} transcripts x {transcript_counts.shape[1]} samples")
print(f"Gene-level matrix: {gene_counts.shape[0]:,} genes x {gene_counts.shape[1]} samples")

**Coding vs non-coding genes and transcripts**

In [ ]:
gene_biotype_series = pd.Series(gene_biotype)
transcript_biotype_series = pd.Series(transcript_biotype)

gene_class = gene_biotype_series.map(lambda b: "Protein-coding" if b == "protein_coding" else "Non-coding")
transcript_class = transcript_biotype_series.map(lambda b: "Protein-coding" if b == "protein_coding" else "Non-coding")

biotype_bar_data = pd.DataFrame({
    "Protein-coding": [(gene_class == "Protein-coding").sum(), (transcript_class == "Protein-coding").sum()],
    "Non-coding": [(gene_class == "Non-coding").sum(), (transcript_class == "Non-coding").sum()],
}, index=["Genes", "Transcripts"])

detailed_gene_biotypes = gene_biotype_series.value_counts().rename("Gene count").to_frame()
detailed_transcript_biotypes = transcript_biotype_series.value_counts().rename("Transcript count").to_frame()

fig, ax = plt.subplots(figsize=(7.5, 5.5))
x = np.arange(len(biotype_bar_data.index))
width = 0.32
colors_biotype = {"Protein-coding": "#1B4F72", "Non-coding": "#B03A2E"}

for i, column in enumerate(biotype_bar_data.columns):
    values = biotype_bar_data[column].values
    bars = ax.bar(x + (i - 0.5) * width, values, width, label=column, color=colors_biotype[column])
    for bar, value in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, value + max(biotype_bar_data.values.flatten()) * 0.01, f"{value:,}", ha="center", fontsize=10.5, fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels(biotype_bar_data.index)
ax.set_ylabel("Count")
ax.set_title("Coding vs non-coding genes and transcripts")
ax.legend(frameon=False)
bold_ticks(ax)

plt.tight_layout()
plt.savefig(FIGDIR / "08_biotype_composition.png", bbox_inches="tight")
plt.show()

**Export gene/transcript biotype composition to Excel**

In [ ]:
biotype_excel_path = TABLEDIR / "gene_transcript_biotype_composition.xlsx"

with pd.ExcelWriter(biotype_excel_path, engine="openpyxl") as writer:
    biotype_bar_data.to_excel(writer, sheet_name="summary")
    detailed_gene_biotypes.to_excel(writer, sheet_name="gene_biotypes_detailed")
    detailed_transcript_biotypes.to_excel(writer, sheet_name="transcript_biotypes_detailed")

biotype_excel_path

**Principal Component Analysis at transcript level and gene level**

In [ ]:
from sklearn.decomposition import PCA

def compute_pca(expression_df, n_top=2000):
    log_expr = np.log2(expression_df + 1)
    top_features = log_expr.var(axis=1).sort_values(ascending=False).head(n_top).index
    matrix = log_expr.loc[top_features].T.values
    matrix = matrix - matrix.mean(axis=0)
    pca_model = PCA(n_components=2, random_state=0)
    scores = pca_model.fit_transform(matrix)
    return scores, pca_model.explained_variance_ratio_

pca_transcript_scores, pca_transcript_var = compute_pca(transcript_tpm)
pca_gene_scores, pca_gene_var = compute_pca(gene_tpm)


def draw_group_region(ax, x, y, color):
    if len(x) < 2:
        return
    cx, cy = np.mean(x), np.mean(y)
    width = (np.max(x) - np.min(x)) + 14
    height = (np.max(y) - np.min(y)) + 14
    region = Ellipse((cx, cy), width, height, facecolor=color, alpha=0.12, edgecolor=color, linewidth=1.8, linestyle="--", zorder=1)
    ax.add_patch(region)


fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

for ax, scores, var_ratio, title in [
    (axes[0], pca_transcript_scores, pca_transcript_var, "Transcript level"),
    (axes[1], pca_gene_scores, pca_gene_var, "Gene level"),
]:
    for group in GROUP_COLOR:
        idx = [i for i, s in enumerate(SAMPLES) if GROUP[s] == group]
        draw_group_region(ax, scores[idx, 0], scores[idx, 1], GROUP_COLOR[group])

    for i, sample in enumerate(SAMPLES):
        ax.scatter(scores[i, 0], scores[i, 1], color=SAMPLE_COLOR[sample], s=140, edgecolor="white", linewidth=1.2, zorder=3)
        ax.annotate(sample, (scores[i, 0], scores[i, 1]), textcoords="offset points", xytext=(7, 6), fontsize=10, fontweight="bold", zorder=4)
    ax.set_xlabel(f"PC1 ({var_ratio[0] * 100:.1f}%)")
    ax.set_ylabel(f"PC2 ({var_ratio[1] * 100:.1f}%)")
    ax.set_title(title)
    ax.axhline(0, color="#CCCCCC", lw=1, zorder=1)
    ax.axvline(0, color="#CCCCCC", lw=1, zorder=1)
    bold_ticks(ax)

axes[1].legend(
    handles=[plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=GROUP_COLOR[g], markersize=11) for g in GROUP_COLOR],
    labels=list(GROUP_COLOR.keys()),
    loc="upper center",
    bbox_to_anchor=(-0.15, -0.14),
    ncol=2,
    frameon=False,
)

plt.tight_layout()
plt.savefig(FIGDIR / "09_pca.png", bbox_inches="tight")
plt.show()

**Gene-level differential expression (PyDESeq2): Ngousso colony vs Field (Akoda)**

In [ ]:
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

GROUP_SHORT = {"Ngousso colony": "Ngousso", "Field (Akoda)": "Field"}

deseq_metadata = pd.DataFrame({"condition": [GROUP_SHORT[GROUP[s]] for s in SAMPLES]}, index=SAMPLES)
deseq_counts = gene_counts.T.loc[SAMPLES]
deseq_counts = deseq_counts.loc[:, deseq_counts.sum(axis=0) > 0]

dds = DeseqDataSet(counts=deseq_counts, metadata=deseq_metadata, design="~condition", refit_cooks=True, quiet=True)
dds.deseq2()

stat_res = DeseqStats(dds, contrast=["condition", "Field", "Ngousso"], quiet=True)
stat_res.summary()

de_results = stat_res.results_df.copy()
de_results.index.name = "gene_id"
de_results["significant"] = (de_results["padj"] < 0.05) & (de_results["log2FoldChange"].abs() > 1)

print(f"Genes tested: {de_results.shape[0]:,}")
print(f"Significant genes (padj<0.05, |log2FC|>1): {int(de_results['significant'].sum()):,}")

**Gene name and functional annotation (NCBI Gene, cross-referenced through the RefSeq transcript accession recorded in the AcolN3 GTF)**

In [ ]:
ANNOTATION_PATH = Path("/mnt/hpc_acegid/home/khadmig/work/script/For_Lynda/anovec-rnaseq-nf/references/gene_annotations.tsv")

if ANNOTATION_PATH.exists():
    gene_annotation = pd.read_csv(ANNOTATION_PATH, sep="\t", dtype=str).set_index("gene_id")
    gene_annotation = gene_annotation.reindex(de_results.index)
else:
    gene_annotation = pd.DataFrame(index=de_results.index, columns=["accession", "gene_name", "description", "function"])

gene_annotation = gene_annotation.fillna("")


def gene_label(gene_id):
    name = gene_annotation.loc[gene_id, "gene_name"] if gene_id in gene_annotation.index else ""
    return name if name else gene_id


annotated_count = int((gene_annotation["gene_name"] != "").sum())
print(f"Significant genes with an NCBI gene name: {annotated_count:,} / {gene_annotation.shape[0]:,}")

**MA plot (log fold change vs mean expression) at gene level**

In [ ]:
not_sig = de_results[~de_results["significant"]]
up_in_field = de_results[de_results["significant"] & (de_results["log2FoldChange"] > 0)]
up_in_ngousso = de_results[de_results["significant"] & (de_results["log2FoldChange"] < 0)]

DE_COLORS = {
    "Not significant": "#9E9E9E",
    "Up in Field (Akoda)": GROUP_COLOR["Field (Akoda)"],
    "Up in Ngousso colony": GROUP_COLOR["Ngousso colony"],
}

fig, ax = plt.subplots(figsize=(9, 6.5))

ax.scatter(np.log10(not_sig["baseMean"] + 1), not_sig["log2FoldChange"], s=8, color=DE_COLORS["Not significant"], alpha=0.5, label="Not significant")
ax.scatter(np.log10(up_in_field["baseMean"] + 1), up_in_field["log2FoldChange"], s=14, color=DE_COLORS["Up in Field (Akoda)"], alpha=0.85, label="Up in Field (Akoda)")
ax.scatter(np.log10(up_in_ngousso["baseMean"] + 1), up_in_ngousso["log2FoldChange"], s=14, color=DE_COLORS["Up in Ngousso colony"], alpha=0.85, label="Up in Ngousso colony")

ax.axhline(0, color="black", lw=1)
ax.axhline(1, color="#4B4B4B", lw=1, linestyle="--", alpha=0.5)
ax.axhline(-1, color="#4B4B4B", lw=1, linestyle="--", alpha=0.5)
ax.set_xlabel("log10(mean normalized count + 1)")
ax.set_ylabel("log2 fold change (Field vs Ngousso)")
ax.set_title("Gene-level differential expression: MA plot")
ax.legend(frameon=False, loc="center left", bbox_to_anchor=(1.02, 0.5))

summary_text = (
    f"Not significant: {len(not_sig):,}\n"
    f"Up in Field (Akoda): {len(up_in_field):,}\n"
    f"Up in Ngousso colony: {len(up_in_ngousso):,}\n"
    f"Threshold: padj < 0.05, |log2FC| > 1"
)
ax.text(1.02, 0.02, summary_text, transform=ax.transAxes, fontsize=9.5, va="bottom", ha="left")
bold_ticks(ax)

plt.tight_layout()
plt.savefig(FIGDIR / "10_gene_ma_plot.png", bbox_inches="tight")
plt.show()

**Volcano plot at gene level**

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

ax.scatter(not_sig["log2FoldChange"], -np.log10(not_sig["padj"].clip(lower=1e-300)), s=8, color=DE_COLORS["Not significant"], alpha=0.5, label="Not significant")
ax.scatter(up_in_field["log2FoldChange"], -np.log10(up_in_field["padj"].clip(lower=1e-300)), s=14, color=DE_COLORS["Up in Field (Akoda)"], alpha=0.85, label="Up in Field (Akoda)")
ax.scatter(up_in_ngousso["log2FoldChange"], -np.log10(up_in_ngousso["padj"].clip(lower=1e-300)), s=14, color=DE_COLORS["Up in Ngousso colony"], alpha=0.85, label="Up in Ngousso colony")

ax.axvline(1, color="#4B4B4B", lw=1, linestyle="--", alpha=0.5)
ax.axvline(-1, color="#4B4B4B", lw=1, linestyle="--", alpha=0.5)
ax.axhline(-np.log10(0.05), color="#4B4B4B", lw=1, linestyle="--", alpha=0.5)
ax.set_xlabel("log2 fold change (Field vs Ngousso)")
ax.set_ylabel("-log10(padj)")
ax.set_title("Gene-level differential expression: volcano plot")
ax.legend(frameon=False, loc="center left", bbox_to_anchor=(1.02, 0.5))

summary_text = (
    f"Not significant: {len(not_sig):,}\n"
    f"Up in Field (Akoda): {len(up_in_field):,}\n"
    f"Up in Ngousso colony: {len(up_in_ngousso):,}\n"
    f"Threshold: padj < 0.05, |log2FC| > 1"
)
ax.text(1.02, 0.02, summary_text, transform=ax.transAxes, fontsize=9.5, va="bottom", ha="left")
bold_ticks(ax)

plt.tight_layout()
plt.savefig(FIGDIR / "11_gene_volcano_plot.png", bbox_inches="tight")
plt.show()

**Heatmap of the top differentially expressed genes**

In [ ]:
N_TOP_GENES = 30

top_de_genes = de_results[de_results["significant"]].sort_values("padj").head(N_TOP_GENES).index
heatmap_log_tpm = np.log2(gene_tpm.loc[top_de_genes, SAMPLES] + 1)
heatmap_zscore = heatmap_log_tpm.sub(heatmap_log_tpm.mean(axis=1), axis=0).div(heatmap_log_tpm.std(axis=1).replace(0, 1), axis=0)

fig, ax = plt.subplots(figsize=(8, max(6, 0.32 * len(top_de_genes))))
im = ax.imshow(heatmap_zscore.values, cmap="RdBu_r", vmin=-2, vmax=2, aspect="auto")

top_de_gene_labels = [gene_label(gene_id) for gene_id in top_de_genes]

ax.set_xticks(range(len(SAMPLES)))
ax.set_xticklabels(SAMPLES, rotation=45, ha="right")
ax.set_yticks(range(len(top_de_genes)))
ax.set_yticklabels(top_de_gene_labels, fontsize=8)

for label, sample in zip(ax.get_xticklabels(), SAMPLES):
    label.set_color(SAMPLE_COLOR[sample])

for i in range(heatmap_log_tpm.shape[0]):
    for j in range(heatmap_log_tpm.shape[1]):
        value = heatmap_log_tpm.values[i, j]
        z = heatmap_zscore.values[i, j]
        text_color = "white" if abs(z) > 1.2 else "black"
        ax.text(j, i, f"{value:.1f}", ha="center", va="center", color=text_color, fontsize=6.5)

bold_ticks(ax)
ax.set_title(f"Top {len(top_de_genes)} differentially expressed genes (log2 TPM, row z-score)")
cbar = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.03)
cbar.set_label("Row z-score", fontweight="bold")

plt.tight_layout()
plt.savefig(FIGDIR / "14_top_de_genes_heatmap.png", bbox_inches="tight")
plt.show()

**Most highly expressed genes within each group's significant set, gene name size proportional to mean expression in that group**

In [ ]:
WORDCLOUD_TOP_N = 35
MIN_FONT = 9
MAX_FONT = 40


def scale_fontsize(values):
    values = np.asarray(values, dtype=float)
    if values.max() == values.min():
        return np.full(values.shape, (MIN_FONT + MAX_FONT) / 2)
    scaled = (values - values.min()) / (values.max() - values.min())
    return MIN_FONT + scaled * (MAX_FONT - MIN_FONT)


def place_gene_labels(ax, renderer, labels, sizes, color):
    order = np.argsort(-np.asarray(sizes))
    placed_boxes = []
    for rank, idx in enumerate(order):
        label = labels[idx]
        fontsize = sizes[idx]
        placed = False
        x, y = 0.5, 0.5
        for attempt in range(150):
            radius = 0.02 * attempt
            if radius > 0.85:
                break
            angle = attempt * 2.399963
            x = 0.5 + radius * np.cos(angle)
            y = 0.5 + radius * np.sin(angle) * 0.8
            text_obj = ax.text(x, y, label, fontsize=fontsize, color=color, ha="center", va="center",
                                fontweight="bold" if rank < 5 else "normal")
            bbox_data = text_obj.get_window_extent(renderer=renderer).transformed(ax.transData.inverted())
            padded = bbox_data.expanded(1.05, 1.2)
            overlap = any(padded.overlaps(existing) for existing in placed_boxes)
            out_of_bounds = padded.x0 < 0.01 or padded.x1 > 0.99 or padded.y0 < 0.01 or padded.y1 > 0.99
            if not overlap and not out_of_bounds:
                placed_boxes.append(padded)
                placed = True
                break
            text_obj.remove()
        if not placed:
            ax.text(0.5, 0.5, label, fontsize=fontsize * 0.55, color=color, alpha=0.35, ha="center", va="center")


def group_wordcloud_genes(de_subset, group_samples):
    mean_expression = gene_tpm.loc[de_subset.index, group_samples].mean(axis=1)
    top_genes = mean_expression.sort_values(ascending=False).head(WORDCLOUD_TOP_N)
    labels = [gene_label(gene_id) for gene_id in top_genes.index]
    sizes = scale_fontsize(np.log2(top_genes.values + 1))
    return labels, sizes


fig, axes = plt.subplots(1, 2, figsize=(15, 8))
for ax in axes:
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")
plt.tight_layout()
fig.canvas.draw()
renderer = fig.canvas.get_renderer()

ngousso_labels, ngousso_sizes = group_wordcloud_genes(up_in_ngousso, ngousso_samples)
field_labels, field_sizes = group_wordcloud_genes(up_in_field, field_samples)

for ax, labels, sizes, title, color in [
    (axes[0], ngousso_labels, ngousso_sizes, "Up in Ngousso colony", GROUP_COLOR["Ngousso colony"]),
    (axes[1], field_labels, field_sizes, "Up in Field (Akoda)", GROUP_COLOR["Field (Akoda)"]),
]:
    place_gene_labels(ax, renderer, labels, sizes, color)
    ax.set_title(title, color=color)

fig.suptitle(
    f"Top {WORDCLOUD_TOP_N} significant genes per group, text size proportional to mean TPM in that group\n\n",
    fontsize=14, fontweight="bold", y=1.1,
)

plt.savefig(FIGDIR / "15_gene_expression_wordcloud.png", bbox_inches="tight")
plt.show()

In [ ]:
WORDCLOUD_TOP_N = 35
MIN_FONT = 9
MAX_FONT = 40


def scale_fontsize(values):
    values = np.asarray(values, dtype=float)

    if values.max() == values.min():
        return np.full(
            values.shape,
            (MIN_FONT + MAX_FONT) / 2
        )

    scaled = (values - values.min()) / (
        values.max() - values.min()
    )

    return MIN_FONT + scaled * (MAX_FONT - MIN_FONT)


def place_gene_labels(ax, renderer, labels, sizes, color):
    order = np.argsort(-np.asarray(sizes))
    placed_boxes = []

    for rank, idx in enumerate(order):
        label = labels[idx]
        fontsize = sizes[idx]
        placed = False

        for attempt in range(250):
            radius = 0.012 * attempt

            if radius > 0.78:
                break

            angle = attempt * 2.399963

            x = 0.5 + radius * np.cos(angle)
            y = 0.5 + radius * np.sin(angle) * 0.82

            text_obj = ax.text(
                x,
                y,
                label,
                fontsize=fontsize,
                color=color,
                ha="center",
                va="center",
                fontweight="bold" if rank < 5 else "normal"
            )

            bbox_data = (
                text_obj
                .get_window_extent(renderer=renderer)
                .transformed(ax.transData.inverted())
            )

            padded = bbox_data.expanded(1.035, 1.12)

            overlap = any(
                padded.overlaps(existing)
                for existing in placed_boxes
            )

            out_of_bounds = (
                padded.x0 < 0.025
                or padded.x1 > 0.975
                or padded.y0 < 0.025
                or padded.y1 > 0.975
            )

            if not overlap and not out_of_bounds:
                placed_boxes.append(padded)
                placed = True
                break

            text_obj.remove()

        if not placed:
            ax.text(
                0.5,
                0.5,
                label,
                fontsize=fontsize * 0.5,
                color=color,
                alpha=0.35,
                ha="center",
                va="center"
            )


def group_wordcloud_genes(de_subset, group_samples):
    mean_expression = gene_tpm.loc[
        de_subset.index,
        group_samples
    ].mean(axis=1)

    top_genes = (
        mean_expression
        .sort_values(ascending=False)
        .head(WORDCLOUD_TOP_N)
    )

    labels = [
        gene_label(gene_id)
        for gene_id in top_genes.index
    ]

    sizes = scale_fontsize(
        np.log2(top_genes.values + 1)
    )

    return labels, sizes


fig, axes = plt.subplots(
    1,
    2,
    figsize=(16, 8),
    gridspec_kw={"wspace": 0.04}
)

fig.patch.set_facecolor("white")

for ax in axes:
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")
    ax.set_facecolor("white")


ngousso_labels, ngousso_sizes = group_wordcloud_genes(
    up_in_ngousso,
    ngousso_samples
)

field_labels, field_sizes = group_wordcloud_genes(
    up_in_field,
    field_samples
)


fig.canvas.draw()
renderer = fig.canvas.get_renderer()


for ax, labels, sizes, title, color in [
    (
        axes[0],
        ngousso_labels,
        ngousso_sizes,
        "Ngousso colony",
        GROUP_COLOR["Ngousso colony"]
    ),
    (
        axes[1],
        field_labels,
        field_sizes,
        "Field (Akoda)",
        GROUP_COLOR["Field (Akoda)"]
    ),
]:

    place_gene_labels(
        ax,
        renderer,
        labels,
        sizes,
        color
    )

    ax.text(
        0.5,
        1.025,
        title,
        transform=ax.transAxes,
        ha="center",
        va="bottom",
        fontsize=18,
        fontweight="bold",
        color=color
    )

    ax.plot(
        [0.30, 0.70],
        [0.995, 0.995],
        transform=ax.transAxes,
        linewidth=3,
        solid_capstyle="round",
        color=color
    )


fig.suptitle(
    "Genes with increased expression in each mosquito group",
    fontsize=19,
    fontweight="bold",
    y=1.1
)

fig.text(
    0.5,
    1.06,
    f"Top {WORDCLOUD_TOP_N} genes shown; label size reflects mean TPM within each group",
    ha="center",
    va="top",
    fontsize=11,
    color="#555555"
)


plt.subplots_adjust(
    left=0.025,
    right=0.975,
    top=0.89,
    bottom=0.035,
    wspace=0.04
)


plt.savefig(
    FIGDIR / "15_gene_expression_wordcloud.png",
    dpi=400,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

**Shared vs group-specific genes**

In [ ]:
gene_detected = gene_tpm >= DETECTION_TPM
gene_ngousso_detected = gene_detected[ngousso_samples].sum(axis=1) >= MIN_REPLICATES
gene_field_detected = gene_detected[field_samples].sum(axis=1) >= MIN_REPLICATES

gene_category = np.select(
    [gene_ngousso_detected & gene_field_detected, gene_ngousso_detected & ~gene_field_detected, ~gene_ngousso_detected & gene_field_detected],
    ["Shared", "Ngousso colony only", "Field (Akoda) only"],
    default="Not detected",
)

gene_table = gene_tpm.round(2).copy()
gene_table.insert(0, "category", gene_category)
gene_table = gene_table.reset_index().rename(columns={"index": "gene_id"})

gene_category_counts = gene_table["category"].value_counts().reindex(category_order).fillna(0).astype(int)
gene_category_counts

**Coding vs non-coding composition of detected genes and transcripts, per group**

In [ ]:
def detected_biotype_counts(detected_ngousso, detected_field, biotype_class_series):
    ng = detected_ngousso.reindex(biotype_class_series.index).fillna(False)
    fi = detected_field.reindex(biotype_class_series.index).fillna(False)
    coding = biotype_class_series == "Protein-coding"
    noncoding = biotype_class_series == "Non-coding"
    return pd.DataFrame({
        "Ngousso colony": [int((coding & ng).sum()), int((noncoding & ng).sum())],
        "Field (Akoda)": [int((coding & fi).sum()), int((noncoding & fi).sum())],
    }, index=["Protein-coding", "Non-coding"])

gene_group_biotype = detected_biotype_counts(gene_ngousso_detected, gene_field_detected, gene_class)
transcript_group_biotype = detected_biotype_counts(ngousso_detected, field_detected, transcript_class)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

for ax, data, level_name in [(axes[0], gene_group_biotype, "Genes"), (axes[1], transcript_group_biotype, "Transcripts")]:
    x = np.arange(len(data.index))
    width = 0.32
    for i, group in enumerate(data.columns):
        values = data[group].values
        bars = ax.bar(x + (i - 0.5) * width, values, width, label=group, color=GROUP_COLOR[group])
        for bar, value in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width() / 2, value + data.values.max() * 0.015, f"{value:,}", ha="center", fontsize=10, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(data.index)
    ax.set_ylabel("Detected count")
    ax.set_title(level_name)
    bold_ticks(ax)

axes[0].legend(frameon=False, loc="upper center", bbox_to_anchor=(1.1, -0.14), ncol=2)

fig.suptitle("Detected coding vs non-coding features per group", fontsize=15, fontweight="bold", y=1.02)

plt.tight_layout()
plt.savefig(FIGDIR / "13_biotype_by_group.png", bbox_inches="tight")
plt.show()

**Transcript and gene overlap, side by side**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))

for ax, n_only_a, n_only_b, n_shared, level_name in [
    (axes[0], n_ngousso_only, n_field_only, n_shared, "Transcript level"),
    (axes[1], gene_category_counts["Ngousso colony only"], gene_category_counts["Field (Akoda) only"], gene_category_counts["Shared"], "Gene level"),
]:
    left_circle = Circle((-0.72, 0), 1.35, color=GROUP_COLOR["Ngousso colony"], alpha=0.55, linewidth=2, edgecolor="white")
    right_circle = Circle((0.72, 0), 1.35, color=GROUP_COLOR["Field (Akoda)"], alpha=0.55, linewidth=2, edgecolor="white")
    ax.add_patch(left_circle)
    ax.add_patch(right_circle)

    ax.text(-1.35, 0, f"{n_only_a:,}", ha="center", va="center", fontsize=18, fontweight="bold")
    ax.text(1.35, 0, f"{n_only_b:,}", ha="center", va="center", fontsize=18, fontweight="bold")
    ax.text(0, 0, f"{n_shared:,}", ha="center", va="center", fontsize=18, fontweight="bold")

    ax.text(-0.72, 1.62, "Ngousso colony", ha="center", va="center", fontsize=12.5, fontweight="bold", color=GROUP_COLOR["Ngousso colony"])
    ax.text(0.72, 1.62, "Field (Akoda)", ha="center", va="center", fontsize=12.5, fontweight="bold", color=GROUP_COLOR["Field (Akoda)"])
    ax.set_xlim(-2.4, 2.4)
    ax.set_ylim(-1.7, 2.1)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(level_name, pad=12)

fig.suptitle("Detection overlap: transcript level vs gene level", fontsize=15, fontweight="bold", y=1.02)

plt.tight_layout()
plt.savefig(FIGDIR / "12_transcript_gene_overlap.png", bbox_inches="tight")
plt.show()

**Export transcript-level counts and categories to Excel**

In [ ]:
transcript_excel_path = TABLEDIR / "transcript_counts_analysis.xlsx"

transcript_counts_export = transcript_counts.reset_index().rename(columns={"Name": "transcript_id"})
transcript_group1 = transcript_table[transcript_table["category"] == "Ngousso colony only"][["transcript_id"] + SAMPLES]
transcript_group2 = transcript_table[transcript_table["category"] == "Field (Akoda) only"][["transcript_id"] + SAMPLES]
transcript_shared = transcript_table[transcript_table["category"] == "Shared"][["transcript_id"] + SAMPLES]

transcript_expression_summary = pd.DataFrame({
    "metric": ["Total transcripts", "Shared", "Ngousso colony only", "Field (Akoda) only", "Not detected",
               "Mean TPM (all samples)", "Median TPM (all samples)"],
    "value": [
        transcript_tpm.shape[0],
        category_counts["Shared"],
        category_counts["Ngousso colony only"],
        category_counts["Field (Akoda) only"],
        category_counts["Not detected"],
        transcript_tpm.values.mean(),
        np.median(transcript_tpm.values),
    ],
})

with pd.ExcelWriter(transcript_excel_path, engine="openpyxl") as writer:
    transcript_counts_export.to_excel(writer, sheet_name="counts_matrix", index=False)
    transcript_group1.to_excel(writer, sheet_name="ngousso_only", index=False)
    transcript_group2.to_excel(writer, sheet_name="field_only", index=False)
    transcript_shared.to_excel(writer, sheet_name="shared", index=False)
    transcript_expression_summary.to_excel(writer, sheet_name="summary", index=False)

transcript_excel_path

**Export gene-level counts, categories, and differential expression to Excel**

In [ ]:
gene_excel_path = TABLEDIR / "gene_counts_analysis.xlsx"

gene_counts_export = gene_counts.reset_index().rename(columns={"index": "gene_id"})
gene_group1 = gene_table[gene_table["category"] == "Ngousso colony only"][["gene_id"] + SAMPLES]
gene_group2 = gene_table[gene_table["category"] == "Field (Akoda) only"][["gene_id"] + SAMPLES]
gene_shared = gene_table[gene_table["category"] == "Shared"][["gene_id"] + SAMPLES]

gene_expression_summary = pd.DataFrame({
    "metric": ["Total genes", "Shared", "Ngousso colony only", "Field (Akoda) only", "Not detected",
               "Mean TPM (all samples)", "Median TPM (all samples)",
               "Significant DE genes (padj<0.05, |log2FC|>1)", "Up in Field", "Up in Ngousso"],
    "value": [
        gene_tpm.shape[0],
        gene_category_counts["Shared"],
        gene_category_counts["Ngousso colony only"],
        gene_category_counts["Field (Akoda) only"],
        gene_category_counts["Not detected"],
        gene_tpm.values.mean(),
        np.median(gene_tpm.values),
        int(de_results["significant"].sum()),
        int((de_results["significant"] & (de_results["log2FoldChange"] > 0)).sum()),
        int((de_results["significant"] & (de_results["log2FoldChange"] < 0)).sum()),
    ],
})

de_results_export = de_results.reset_index()
de_results_export = de_results_export.merge(
    gene_annotation[["gene_name", "description", "function"]].reset_index(), on="gene_id", how="left"
)
annotation_columns = ["gene_id", "gene_name", "description", "function"]
de_results_export = de_results_export[annotation_columns + [c for c in de_results_export.columns if c not in annotation_columns]]

de_up = de_results_export[de_results_export["significant"] & (de_results_export["log2FoldChange"] > 0)]
de_down = de_results_export[de_results_export["significant"] & (de_results_export["log2FoldChange"] < 0)]

with pd.ExcelWriter(gene_excel_path, engine="openpyxl") as writer:
    gene_counts_export.to_excel(writer, sheet_name="counts_matrix", index=False)
    gene_group1.to_excel(writer, sheet_name="ngousso_only", index=False)
    gene_group2.to_excel(writer, sheet_name="field_only", index=False)
    gene_shared.to_excel(writer, sheet_name="shared", index=False)
    de_results_export.to_excel(writer, sheet_name="differential_expression", index=False)
    de_up.to_excel(writer, sheet_name="up_in_field", index=False)
    de_down.to_excel(writer, sheet_name="up_in_ngousso", index=False)
    gene_expression_summary.to_excel(writer, sheet_name="summary", index=False)

gene_excel_path